# Regression Analysis

Three regression tasks on the Sheffield Road Collision dataset:

| Task | Goal |
|------|------|
| A | Predict `number_of_casualties` from road and environment features |
| B | Model collision frequency trend over time (monthly aggregation) |
| C | Augment Task A with spatial coordinates; map predicted casualties across Sheffield |

**Log transform:** `number_of_casualties` is right-skewed (most collisions have 1 casualty).
`np.log1p` is applied to the target before training. All metrics are computed after
back-transforming predictions with `np.expm1` so they are reported in original casualty units.

---
# 1. Configuration and Imports

In [ ]:
import os
import pickle
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')


def find_project_root(marker='config.yml'):
    current = Path().resolve()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f'Could not find {marker} in any parent directory')


ROOT_DIR = find_project_root()
NOTEBOOKS_DIR = ROOT_DIR / 'notebooks'

with open(ROOT_DIR / 'config.yml') as f:
    cfg = yaml.safe_load(f)

with open(NOTEBOOKS_DIR / 'notebook-config.yml') as f:
    nb_cfg = yaml.safe_load(f)

NB_CONFIG = {
    'figsize_wide':   nb_cfg['plotting']['figsize_wide'],
    'figsize_square': nb_cfg['plotting']['figsize_square'],
    'figsize_tall':   nb_cfg['plotting']['figsize_tall'],
    'palette':        nb_cfg['plotting']['palette'],
}

sns.set_theme(style='whitegrid', palette=NB_CONFIG['palette'])
print(f'Project root: {ROOT_DIR}')
print('Configs loaded.')

---
# 2. Load Processed Data

In [ ]:
df = pd.read_csv(ROOT_DIR / cfg['preprocessing']['output_path'])

print(f'Shape: {df.shape}')
df.head(3)

---
# 3. Shared Utilities

In [ ]:
def regression_metrics(model_name, y_true_orig, y_pred_orig):
    """
    Compute MAE, RMSE, and R2 in original (casualty count) units.
    All predictions passed in must already be back-transformed from log space.
    """
    mae  = mean_absolute_error(y_true_orig, y_pred_orig)
    rmse = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
    r2   = r2_score(y_true_orig, y_pred_orig)

    print(f'--- {model_name} ---')
    print(f'MAE:  {mae:.4f}')
    print(f'RMSE: {rmse:.4f}')
    print(f'R2:   {r2:.4f}')

    return {'model_name': model_name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}


def plot_residuals(ax_scatter, ax_resid, y_true, y_pred, model_name, color):
    """Predicted vs actual scatter + residual plot on provided axes."""
    residuals = y_true - y_pred

    ax_scatter.scatter(y_true, y_pred, alpha=0.3, s=15, color=color)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax_scatter.plot(lims, lims, 'r--', linewidth=1)
    ax_scatter.set_xlabel('Actual')
    ax_scatter.set_ylabel('Predicted')
    ax_scatter.set_title(f'{model_name}\nPredicted vs Actual', fontsize=9, fontweight='bold')

    ax_resid.scatter(y_pred, residuals, alpha=0.3, s=15, color=color)
    ax_resid.axhline(0, color='red', linestyle='--', linewidth=1)
    ax_resid.set_xlabel('Predicted')
    ax_resid.set_ylabel('Residual')
    ax_resid.set_title(f'{model_name}\nResiduals', fontsize=9, fontweight='bold')

---
# Task A - Predict `number_of_casualties`

**Models:** Linear Regression (baseline), Ridge (regularized baseline),
Random Forest Regressor, XGBoost Regressor.

GridSearchCV is used for RF and XGB. Linear and Ridge are fit directly
(Ridge uses a small alpha grid via GridSearchCV).

## A.1 Feature Selection and Target Distribution

In [ ]:
reg_cfg = cfg['regression']

# Columns that are outcomes, identifiers, or direct derivations of the target
# collision_severity and number_of_vehicles are outcomes, not predictors for casualties
exclude_cols = [
    reg_cfg['target_col'],       # number_of_casualties - this is the target
    'collision_index',           # identifier
    'collision_injury_based',    # outcome derivation
    'local_authority_highway_current',  # high-cardinality string
    'collision_severity',        # outcome variable
    'number_of_vehicles',        # fellow outcome, not a causal predictor
]

target_col = reg_cfg['target_col']
feature_cols = [c for c in df.columns if c not in exclude_cols and df[c].dtype != object]

X = df[feature_cols].copy()
y_raw = df[target_col].copy()

print(f'Target: {target_col}')
print(f'Features ({len(feature_cols)}): {feature_cols}')
print('\nTarget distribution:')
print(y_raw.describe().round(3))
print(f'\nSkewness (raw): {y_raw.skew():.3f}')

## A.2 Log Transform

In [ ]:
# Log-transform the target using log1p (handles zero values safely: log1p(x) = log(1+x))
# Back-transform with np.expm1 (expm1(x) = exp(x) - 1) before computing metrics
y = np.log1p(y_raw)

print(f'Skewness before log1p: {y_raw.skew():.3f}')
print(f'Skewness after log1p:  {y.skew():.3f}')

fig, axes = plt.subplots(1, 2, figsize=tuple(NB_CONFIG['figsize_wide']))
axes[0].hist(y_raw, bins=20, color=sns.color_palette(NB_CONFIG['palette'])[0], edgecolor='white')
axes[0].set_title('Target Distribution - Raw', fontweight='bold')
axes[0].set_xlabel('number_of_casualties')

axes[1].hist(y, bins=20, color=sns.color_palette(NB_CONFIG['palette'])[1], edgecolor='white')
axes[1].set_title('Target Distribution - log1p Transformed', fontweight='bold')
axes[1].set_xlabel('log1p(number_of_casualties)')

plt.tight_layout()
plt.show()

## A.3 Train / Test Split and Scaling

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=reg_cfg['test_size'],
    random_state=reg_cfg['random_state'],
)

# Keep raw test values for back-transformed metric computation
y_test_orig = np.expm1(y_test)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
cv = KFold(
    n_splits=reg_cfg['cv_folds'],
    shuffle=True,
    random_state=reg_cfg['random_state'],
)

## A.4 Linear Regression Baseline

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred_lr_log  = lr.predict(X_test_scaled)
y_pred_lr_orig = np.expm1(y_pred_lr_log)

results_a = {}
results_a['Linear Regression'] = regression_metrics('Linear Regression', y_test_orig, y_pred_lr_orig)
results_a['Linear Regression']['y_pred_orig'] = y_pred_lr_orig
results_a['Linear Regression']['model'] = lr

## A.5 Ridge Regression

In [ ]:
# Ridge adds L2 regularisation - helps when features are correlated
ridge_param_grid = {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}

ridge_search = GridSearchCV(
    Ridge(),
    ridge_param_grid,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
)
ridge_search.fit(X_train_scaled, y_train)

print(f'Best alpha: {ridge_search.best_params_["alpha"]}')
print(f'Best CV RMSE: {-ridge_search.best_score_:.4f} (log scale)')

ridge_model = ridge_search.best_estimator_
y_pred_ridge_log  = ridge_model.predict(X_test_scaled)
y_pred_ridge_orig = np.expm1(y_pred_ridge_log)

results_a['Ridge'] = regression_metrics('Ridge', y_test_orig, y_pred_ridge_orig)
results_a['Ridge']['y_pred_orig'] = y_pred_ridge_orig
results_a['Ridge']['model'] = ridge_model

## A.6 Random Forest Regressor

In [ ]:
rf_cfg = cfg['supervised']['models']['random_forest']

param_grid_rf = {
    'n_estimators':      rf_cfg['n_estimators'],
    'max_depth':         rf_cfg['max_depth'],
    'min_samples_split': rf_cfg['min_samples_split'],
}

search_rf = GridSearchCV(
    RandomForestRegressor(random_state=reg_cfg['random_state'], n_jobs=-1),
    param_grid_rf,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

print('Running GridSearchCV for Random Forest Regressor...')
search_rf.fit(X_train, y_train)

print(f'Best params:   {search_rf.best_params_}')
print(f'Best CV RMSE:  {-search_rf.best_score_:.4f} (log scale)')

rf_model = search_rf.best_estimator_
y_pred_rf_log  = rf_model.predict(X_test)
y_pred_rf_orig = np.expm1(y_pred_rf_log)

results_a['Random Forest'] = regression_metrics('Random Forest', y_test_orig, y_pred_rf_orig)
results_a['Random Forest']['y_pred_orig'] = y_pred_rf_orig
results_a['Random Forest']['model'] = rf_model
results_a['Random Forest']['best_params'] = search_rf.best_params_

## A.7 XGBoost Regressor

In [ ]:
xgb_cfg = cfg['supervised']['models']['xgboost']

param_grid_xgb = {
    'learning_rate': xgb_cfg['learning_rate'],
    'max_depth':     xgb_cfg['max_depth'],
    'n_estimators':  xgb_cfg['n_estimators'],
    'subsample':     xgb_cfg['subsample'],
}

search_xgb = GridSearchCV(
    XGBRegressor(
        objective='reg:squarederror',
        random_state=reg_cfg['random_state'],
        n_jobs=-1,
        verbosity=0,
    ),
    param_grid_xgb,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

print('Running GridSearchCV for XGBoost Regressor...')
search_xgb.fit(X_train, y_train)

print(f'Best params:  {search_xgb.best_params_}')
print(f'Best CV RMSE: {-search_xgb.best_score_:.4f} (log scale)')

xgb_model = search_xgb.best_estimator_
y_pred_xgb_log  = xgb_model.predict(X_test)
y_pred_xgb_orig = np.expm1(y_pred_xgb_log)

results_a['XGBoost'] = regression_metrics('XGBoost', y_test_orig, y_pred_xgb_orig)
results_a['XGBoost']['y_pred_orig'] = y_pred_xgb_orig
results_a['XGBoost']['model'] = xgb_model
results_a['XGBoost']['best_params'] = search_xgb.best_params_

## A.8 Comparison and Diagnostics

In [ ]:
comparison_a = pd.DataFrame([
    {'Model': name, 'MAE': r['MAE'], 'RMSE': r['RMSE'], 'R2': r['R2']}
    for name, r in results_a.items()
]).set_index('Model').round(4)

print('Task A - Model comparison (metrics in original casualty units):')
comparison_a

In [ ]:
palette = sns.color_palette(NB_CONFIG['palette'], len(results_a))
fig, axes = plt.subplots(2, len(results_a), figsize=(16, 8))

for col, ((model_name, r), color) in enumerate(zip(results_a.items(), palette, strict=False)):
    plot_residuals(
        axes[0][col], axes[1][col],
        y_test_orig, r['y_pred_orig'],
        model_name, color
    )

fig.suptitle('Task A - Predicted vs Actual and Residuals (original units)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importances from the best tree-based model
best_model_name = comparison_a['RMSE'].idxmin()
print(f'Best model by RMSE: {best_model_name}')

best_model = results_a[best_model_name]['model']

if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(
        best_model.feature_importances_,
        index=X_train.columns
    ).sort_values(ascending=True).tail(15)

    fig, ax = plt.subplots(figsize=tuple(NB_CONFIG['figsize_square']))
    ax.barh(importances.index, importances.values, # type: ignore
            color=sns.color_palette(NB_CONFIG['palette'])[0])
    ax.axvline(importances.mean(), color='red', linestyle='--', linewidth=1, label='Mean')
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'{best_model_name} - Top 15 Feature Importances (Task A)',
                 fontsize=12, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()

---
# Task B - Collision Frequency Trend Over Time

Monthly collision counts are aggregated and modelled with Linear Regression
on `collision_year` and `month`. Statistical significance of the trend is
assessed via `scipy.stats.linregress` (p-value at alpha=0.05).

In [ ]:
# Aggregate collision counts by year and month
monthly = (
    df.groupby(['collision_year', 'month'])
    .size()
    .reset_index(name='collision_count')
)

# Create a single time index for plotting (year + month/12)
monthly['time_index'] = monthly['collision_year'] + (monthly['month'] - 1) / 12

print(f'Monthly aggregation shape: {monthly.shape}')
monthly.head()

In [ ]:
X_trend = monthly[['collision_year', 'month']].values
y_trend = monthly['collision_count'].values

lr_trend = LinearRegression()
lr_trend.fit(X_trend, y_trend) # type: ignore
y_trend_pred = lr_trend.predict(X_trend)

# Statistical significance via scipy linregress on time_index
slope, intercept, r_value, p_value, std_err = stats.linregress(
    monthly['time_index'], monthly['collision_count']
)

print(f'Trend slope:    {slope:.4f} collisions/month')
print(f'R2:             {r_value**2:.4f}') # type: ignore
print(f'p-value:        {p_value:.4f}')
print(f'Trend direction: {"increasing" if slope > 0 else "decreasing"}' # type: ignore
      f' ({"statistically significant" if p_value < 0.05 else "not statistically significant"} at p<0.05)') # type: ignore

In [ ]:
fig, ax = plt.subplots(figsize=tuple(NB_CONFIG['figsize_wide']))

ax.plot(monthly['time_index'], monthly['collision_count'],
        alpha=0.6, linewidth=1, label='Actual monthly collisions',
        color=sns.color_palette(NB_CONFIG['palette'])[0])
ax.plot(monthly['time_index'], y_trend_pred,
        linewidth=2, linestyle='--', label='Linear trend',
        color=sns.color_palette(NB_CONFIG['palette'])[2])

ax.set_xlabel('Year')
ax.set_ylabel('Collision count')
ax.set_title(
    f'Task B - Collision Frequency Trend Over Time\n'
    f'Slope: {slope:.3f} collisions/month  |  '
    f'p={p_value:.4f}  |  R2={r_value**2:.4f}', # type: ignore
    fontsize=12, fontweight='bold'
)
ax.legend()
plt.tight_layout()
plt.show()

---
# Task C - Spatial Augmentation

The Task A feature set is augmented with spatial coordinates expressed as
km offsets from Sheffield city centre (OS grid centroid ~435000E, ~387000N).
The best RF and XGB hyperparameters from Task A are reused to isolate the
effect of the spatial features rather than re-tuning.

## C.1 Build Augmented Feature Set

In [ ]:
# Sheffield OS grid centroid (approximate city centre)
SHEFFIELD_EASTING  = 435000
SHEFFIELD_NORTHING = 387000

# Augment Task A feature set with spatial coordinates as km offsets from centroid
X_spatial = X.copy()
X_spatial['easting_km']  = (df['location_easting_osgr']  - SHEFFIELD_EASTING)  / 1000
X_spatial['northing_km'] = (df['location_northing_osgr'] - SHEFFIELD_NORTHING) / 1000

print(f'Task C feature set: {list(X_spatial.columns)}')
print('Added: easting_km, northing_km (km offset from Sheffield city centre)')

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_spatial, y,
    test_size=reg_cfg['test_size'],
    random_state=reg_cfg['random_state'],
)

y_test_c_orig = np.expm1(y_test_c)

print(f'Train: {X_train_c.shape}  |  Test: {X_test_c.shape}')

## C.2 Random Forest + Spatial

In [ ]:
# Reuse best params from Task A RF to keep comparison clean
best_rf_params = results_a['Random Forest'].get('best_params', {})

rf_spatial = RandomForestRegressor(
    **best_rf_params,
    random_state=reg_cfg['random_state'],
    n_jobs=-1,
)
rf_spatial.fit(X_train_c, y_train_c)

y_pred_rf_c_log  = rf_spatial.predict(X_test_c)
y_pred_rf_c_orig = np.expm1(y_pred_rf_c_log)

results_c = {}
results_c['RF + Spatial'] = regression_metrics('RF + Spatial', y_test_c_orig, y_pred_rf_c_orig)
results_c['RF + Spatial']['y_pred_orig'] = y_pred_rf_c_orig
results_c['RF + Spatial']['model'] = rf_spatial

## C.3 XGBoost + Spatial

In [ ]:
best_xgb_params = results_a['XGBoost'].get('best_params', {})

xgb_spatial = XGBRegressor(
    **best_xgb_params,
    objective='reg:squarederror',
    random_state=reg_cfg['random_state'],
    n_jobs=-1,
    verbosity=0,
)
xgb_spatial.fit(X_train_c, y_train_c)

y_pred_xgb_c_log  = xgb_spatial.predict(X_test_c)
y_pred_xgb_c_orig = np.expm1(y_pred_xgb_c_log)

results_c['XGB + Spatial'] = regression_metrics('XGB + Spatial', y_test_c_orig, y_pred_xgb_c_orig)
results_c['XGB + Spatial']['y_pred_orig'] = y_pred_xgb_c_orig
results_c['XGB + Spatial']['model'] = xgb_spatial

## C.4 Comparison Against Task A Baselines

In [ ]:
# Compare spatial-augmented models against their Task A counterparts
comparison_c = pd.DataFrame([
    {'Model': 'Random Forest (Task A)',
     'MAE':  results_a['Random Forest']['MAE'],
     'RMSE': results_a['Random Forest']['RMSE'],
     'R2':   results_a['Random Forest']['R2']},
    {'Model': 'RF + Spatial',
     'MAE':  results_c['RF + Spatial']['MAE'],
     'RMSE': results_c['RF + Spatial']['RMSE'],
     'R2':   results_c['RF + Spatial']['R2']},
    {'Model': 'XGBoost (Task A)',
     'MAE':  results_a['XGBoost']['MAE'],
     'RMSE': results_a['XGBoost']['RMSE'],
     'R2':   results_a['XGBoost']['R2']},
    {'Model': 'XGB + Spatial',
     'MAE':  results_c['XGB + Spatial']['MAE'],
     'RMSE': results_c['XGB + Spatial']['RMSE'],
     'R2':   results_c['XGB + Spatial']['R2']},
]).set_index('Model').round(4)

print('Task C vs Task A - does spatial augmentation help?')
comparison_c

## C.5 Spatial Prediction Map

In [ ]:
# Spatial prediction map - predicted casualties at each collision location
# Use the better spatial model for the map
best_spatial_name = comparison_c.loc[['RF + Spatial', 'XGB + Spatial'], 'RMSE'].idxmin()
best_spatial_model = results_c[best_spatial_name]['model']

# Predict over full dataset for map coverage
y_map_log  = best_spatial_model.predict(X_spatial)
y_map_orig = np.expm1(y_map_log)

fig, ax = plt.subplots(figsize=tuple(NB_CONFIG['figsize_square']))

scatter = ax.scatter(
    (df['location_easting_osgr']  - SHEFFIELD_EASTING)  / 1000,
    (df['location_northing_osgr'] - SHEFFIELD_NORTHING) / 1000,
    c=y_map_orig,
    cmap='YlOrRd',
    alpha=0.4,
    s=8,
)
plt.colorbar(scatter, ax=ax, label='Predicted casualties')
ax.set_xlabel('Easting offset from city centre (km)')
ax.set_ylabel('Northing offset from city centre (km)')
ax.set_title(
    f'Task C - Predicted Casualty Count by Location ({best_spatial_name})',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

---
# Persist Regression Outputs

In [ ]:
output_dir = NOTEBOOKS_DIR / 'stage_outputs'
os.makedirs(output_dir, exist_ok=True)

output_path = output_dir / 'regression.pkl'
with open(output_path, 'wb') as f:
    pickle.dump({
        'task_a': {
            'results':       {
                name: {
                    'MAE':        r['MAE'],
                    'RMSE':       r['RMSE'],
                    'R2':         r['R2'],
                    'y_pred_orig': r['y_pred_orig'],
                    'model':      r['model'],
                    'best_params': r.get('best_params', None),
                }
                for name, r in results_a.items()
            },
            'y_test_orig':   y_test_orig,
            'X_test':        X_test,
            'comparison_df': comparison_a,
            'feature_cols':  list(X_train.columns),
            'scaler':        scaler,
        },
        'task_b': {
            'slope':         slope,
            'intercept':     intercept,
            'p_value':       p_value,
            'r_squared':     r_value ** 2, # type: ignore
            'monthly_df':    monthly,
            'y_trend_pred':  y_trend_pred,
        },
        'task_c': {
            'results':       {
                name: {
                    'MAE':        r['MAE'],
                    'RMSE':       r['RMSE'],
                    'R2':         r['R2'],
                    'y_pred_orig': r['y_pred_orig'],
                    'model':      r['model'],
                }
                for name, r in results_c.items()
            },
            'y_test_orig':   y_test_c_orig,
            'X_test':        X_test_c,
            'comparison_df': comparison_c,
        },
    }, f)

print(f'Regression outputs saved to: {output_path}')